# AVI API Demo

This notebook demonstrates how to use the AVI API as a client.

## Prerequisites

Start the AVI server:
```bash
docker compose up --build
# or
uvicorn main:app --host 0.0.0.0 --port 8000
```

In [ ]:
import requests
import json

# API Configuration
API_URL = "http://localhost:8000"

# Optional: Set your API key if authentication is enabled
API_KEY = None  # "avi_your_key_here"

headers = {}
if API_KEY:
    headers["X-API-Key"] = API_KEY

## 1. Health Check

In [ ]:
response = requests.get(f"{API_URL}/api/v1/health", headers=headers)
print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## 2. Send a Query

In [ ]:
# Safe query example
query_data = {
    "query": "What is machine learning?",
    "use_rag": True,
    "use_safety": True
}

response = requests.post(
    f"{API_URL}/api/v1/query",
    json=query_data,
    headers=headers
)

print(f"Status: {response.status_code}")
result = response.json()
print(f"Response: {result.get('response', 'No response')}")
print(f"Filtered: {result.get('filtered', False)}")
print(f"Safety Score: {result.get('safety_score', 'N/A')}")

## 3. Test Content Filtering

In [ ]:
# Potentially unsafe query (will be filtered)
unsafe_query = {
    "query": "How to hack into a computer system?",
    "use_rag": True,
    "use_safety": True
}

response = requests.post(
    f"{API_URL}/api/v1/query",
    json=unsafe_query,
    headers=headers
)

result = response.json()
print(f"Filtered: {result.get('filtered', False)}")
print(f"Response: {result.get('response', 'Blocked')}")

## 4. Get System Settings

In [ ]:
# Get current RAG settings
response = requests.get(f"{API_URL}/api/v1/settings/rag", headers=headers)
print("RAG Settings:")
print(json.dumps(response.json(), indent=2))

# Get safety settings
response = requests.get(f"{API_URL}/api/v1/settings/safety", headers=headers)
print("\nSafety Settings:")
print(json.dumps(response.json(), indent=2))

## 5. List Filter Rules

In [ ]:
response = requests.get(f"{API_URL}/api/v1/rules", headers=headers)
rules = response.json()

print(f"Total rules: {len(rules)}")

# Show first 5 rules
if rules:
    print("\nSample rules:")
    for rule in rules[:5]:
        print(f"  - [{rule.get('category', 'N/A')}] {rule.get('text', '')[:50]}...")

## 6. Streaming Response (SSE)

In [ ]:
# Streaming query using Server-Sent Events
stream_data = {
    "query": "Explain the benefits of vector databases.",
    "use_rag": True,
    "use_safety": True
}

response = requests.post(
    f"{API_URL}/api/v1/query/stream",
    json=stream_data,
    headers=headers,
    stream=True
)

print("Streaming response:")
for line in response.iter_lines():
    if line:
        decoded = line.decode('utf-8')
        if decoded.startswith('data:'):
            data = decoded[5:].strip()
            if data and data != '[DONE]':
                try:
                    chunk = json.loads(data)
                    print(chunk.get('token', ''), end='', flush=True)
                except json.JSONDecodeError:
                    pass
print()

## 7. Trigger Reindexing

In [ ]:
# Trigger reindexing (requires admin API key)
response = requests.post(f"{API_URL}/api/v1/reindex", headers=headers)
print(f"Reindex status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

## Summary

This demo shows how to:
- Check system health
- Send queries with RAG and safety filtering
- View and manage filter rules
- Use streaming responses
- Trigger reindexing

For full API documentation, visit: http://localhost:8000/docs